# Imports

In [69]:
# Imports
import pandas as pd
from bs4 import BeautifulSoup
import datetime

# Import CSVs

    -Filter to Rajah Caruth Only

In [63]:
# Import CSVs
loopdata = pd.read_csv("noaps_2026_loopdata.csv")
rr_race_results = pd.read_csv("noaps_2026_rr_results.csv")
rr_race_info = pd.read_csv("noaps_2026_rr_race_info.csv")

# Filter to loopdata to just "Rajah Caruth"
raj_loopdata = loopdata[loopdata['driver_name'] == "Rajah Caruth"].copy()

# Merge Racing-Reference
    - rr_race_results + rr_race_info

In [64]:
# Merge both Racing-Reference dfs/CSVs

# Merge rr_race_results with rr_race_info.
# Each race has MANY driver results - - but only ONE race info row.
rr_df = rr_race_results.merge(
    rr_race_info,
    on="race_id",
    how="left",
    validate="many_to_one" # validate="many_to_one" verifies that each race_id has only one
                           # corresponding row in rr_race_info.
)

# Filter to only Rajah
raj_rr = rr_df[rr_df["driver"]=="Rajah Caruth"].copy()

# Prepare Data for Merge
    - Correct Dates from Racing-Reference Data and Loopdata
    - Same Data Formats

In [65]:
# Clean dates - Match the date formates on raj_rr and raj_loopdata
raj_loopdata['race_date'] = pd.to_datetime(raj_loopdata['race_date'])
raj_rr['race_date'] = pd.to_datetime(raj_rr['race_date'])

In [ ]:
#raj_rr
#raj_loopdata

# Simplify the dfs
    - Merge Racing-Reference w/ Loopdata

In [60]:
# Create df of Rajah's  race_date | car_number | sponsor_owner | race_id  from (raj_rr)
raj_rr_car_info = raj_rr[["race_date", "car_number", "sponsor_owner", "points", "track_name", "race_id"]]

# Try Merging (raj_rr_car_info) onto  raj_loopdata
raj_df = raj_loopdata.merge(
    raj_rr_car_info,
    on="race_date",
    how="left",
    validate="one_to_one"
)

**Inspect if All Dates Matched Correctly**

**Not all the Dates Match**

**Inpsect the Missing Dates**

In [73]:
# See how many Car Numbers are missing in the df
print(f"Number of ['car_number']'s na: {raj_df['car_number'].isna().sum()}")

# See what ['track_name'] column name has changed to
print(f"\n {raj_df.columns}")

Number of ['car_number']'s na: 12

 Index(['driver_name', 'driver_id', 'race_date', 'series', 'track_name_x',
       'start', 'mid', 'finish', 'best', 'worst', 'avg',
       'green_flag_passing_diff', 'green_flag_passes',
       'green_flag_times_passed', 'quality_passes', 'pct_quality_passes',
       'fastest_lap', 'top_15_laps', 'pct_top_15_laps', 'laps_led',
       'pct_laps_led', 'laps', 'rating', 'status', 'car_number',
       'sponsor_owner', 'points', 'track_name_y', 'race_id'],
      dtype='object')


In [68]:
# Inspect Specific Race Dates too see How far off they are

    # print Loop Data Dates
print("=== LapRaptor ===")
print(raj_df[
    raj_df["car_number"].isna()
][["race_date", "track_name_x"]].sort_values("race_date").reset_index())

    # Print Race-Reference Dates
print("\n=== Racing-Reference ===")
print(raj_rr_car_info[
    ["race_date", "track_name", "car_number"]
].sort_values("race_date").reset_index())

=== LapRaptor ===
    index  race_date                    track_name_x
0       5 2026-02-15  Daytona International Speedway
1      15 2026-02-22          Atlanta Motor Speedway
2      12 2026-03-08                 Phoenix Raceway
3       2 2026-03-15        Las Vegas Motor Speedway
4       7 2026-03-22              Darlington Raceway
5       3 2026-04-12          Bristol Motor Speedway
6      18 2026-04-19                 Kansas Speedway
7      19 2026-05-24        Charlotte Motor Speedway
8      11 2026-05-31         Nashville Superspeedway
9       0 2026-06-21         San Diego Street Course
10     21 2026-06-28                  Sonoma Raceway
11     20 2026-07-12          Atlanta Motor Speedway

=== Racing-Reference ===
    index  race_date                      track_name  car_number
0       9 2026-02-14  Daytona International Speedway          88
1      45 2026-02-21               EchoPark Speedway          88
2     106 2026-02-28         Circuit of the Americas          88
3     1

**Solution:**

**Nearest Date Merge with a one day Tolerence**

In [74]:
# Sort both DataFrames are sorted by the merge key
raj_loopdata = raj_loopdata.sort_values("race_date").copy()
raj_rr_car_info = raj_rr_car_info.sort_values("race_date").copy()

# Merger each LapRaptor race with ther nearest Racing-Reference race_date
raj_df = pd.merge_asof(
    raj_loopdata,
    raj_rr_car_info,
    on="race_date",
    direction="nearest",
    tolerance=pd.Timedelta("1 day")
)


# Validate Merge
    - to_csv() to prepare for analysis

In [76]:
# Validate Merge / isna()
print(f"Total NAs: {raj_df['car_number'].isna().sum()}")


Total NAs: 0


In [78]:
# Create Rajah Caruth 2026 CSV
raj_df.to_csv("noaps_caruth_2026.csv")